# YOLOv8 Traffic Detection — Test Only

This notebook only tests the pretrained `yolov8n.pt` model on a sample traffic video, displays the annotated output, and prints detections. **No training code is included.**


## 1. Install dependencies
Run once if Ultralytics is not installed.

In [1]:
%pip install -U ultralytics

/home/kcrowly/PycharmProjects/YoloV8-Tumor-Detection/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 2. Load the pretrained model
This uses general-purpose pretrained COCO weights; it is not a custom-trained traffic model.

In [2]:
from ultralytics import YOLO
from pathlib import Path
from IPython.display import Video, display
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

model = YOLO('yolov8n.pt')
print('Model loaded. Detectable classes include:', ', '.join(model.names.values()))

CUDA available: False
Model loaded. Detectable classes include: person, bicycle, car, motorcycle, airplane, bus, train, truck, boat, traffic light, fire hydrant, stop sign, parking meter, bench, bird, cat, dog, horse, sheep, cow, elephant, bear, zebra, giraffe, backpack, umbrella, handbag, tie, suitcase, frisbee, skis, snowboard, sports ball, kite, baseball bat, baseball glove, skateboard, surfboard, tennis racket, bottle, wine glass, cup, fork, knife, spoon, bowl, banana, apple, sandwich, orange, broccoli, carrot, hot dog, pizza, donut, cake, chair, couch, potted plant, bed, dining table, toilet, tv, laptop, mouse, remote, keyboard, cell phone, microwave, oven, toaster, sink, refrigerator, book, clock, vase, scissors, teddy bear, hair drier, toothbrush


## 3. Test on the sample traffic video
Run this cell to detect objects and save an annotated copy of the video. If the URL fails, set `VIDEO_SOURCE` to a local video filename.

In [3]:
VIDEO_SOURCE = 'https://media.geeksforgeeks.org/wp-content/uploads/20240517122022/d.mp4'  # or 'my_traffic_video.mp4'

results = model.predict(
    source=VIDEO_SOURCE,
    save=True,
    imgsz=320,
    conf=0.5,
    project='runs/detect',
    name='traffic_test',
    exist_ok=True,
)
print('Saved predictions under:', Path('runs/detect/traffic_test').resolve())


Found https://media.geeksforgeeks.org/wp-content/uploads/20240517122022/d.mp4 locally at d.mp4


/home/kcrowly/PycharmProjects/YoloV8-Tumor-Detection/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:1174: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/687) /home/kcrowly/PycharmProjects/YoloV8-Tumor-Detection/Notebooks/TEST/d.mp4: 192x320 2 cars, 27.2ms
video 1/1 (frame 2/687) /home/kcrowly/PycharmProjects/YoloV8-Tumor-Detection/Notebooks/TEST/d.mp4: 192x320 3 cars, 14.1ms
video 1/1 (frame 3/687) /home/kcrowly/PycharmProjects/YoloV8-Tumor-Detection/Notebooks/TEST/d.mp4: 192x320 4 cars, 18.7ms
video 1/1 (frame 4/687) /home/kcrowly/PycharmProjects/YoloV8-Tumor-Detection/Notebooks/TEST/d.

## 4. Display the annotated video
The video should show boxes, class labels, and confidence values.

In [4]:
output_dir = Path('runs/detect/traffic_test')
video_files = [p for p in output_dir.glob('*') if p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'}]
if video_files:
    output_video = max(video_files, key=lambda p: p.stat().st_mtime)
    print('Output video:', output_video.resolve())
    display(Video(filename=str(output_video), embed=True))
else:
    print('No video found. Check the previous cell for errors and inspect:', output_dir.resolve())

No video found. Check the previous cell for errors and inspect: /home/kcrowly/PycharmProjects/YoloV8-Tumor-Detection/Notebooks/TEST/runs/detect/traffic_test


## 5. Print detections from the first processed frame
Shows each detected class, confidence score, and bounding-box coordinates.

In [5]:
if results:
    first = results[0]
    if first.boxes is None or len(first.boxes) == 0:
        print('No objects detected above the confidence threshold in the first frame.')
    else:
        for box in first.boxes:
            class_id = int(box.cls.item())
            confidence = float(box.conf.item())
            xyxy = [round(float(v), 1) for v in box.xyxy[0].tolist()]
            print(f'{model.names[class_id]} | confidence={confidence:.3f} | xyxy={xyxy}')
else:
    print('Run the prediction cell first.')

car | confidence=0.815 | xyxy=[339.7, 210.7, 408.7, 264.7]
car | confidence=0.583 | xyxy=[362.5, 155.7, 411.9, 199.4]


## 6. Test your own image or video (optional)
Change `MY_SOURCE` to a path accessible from your notebook, then run the cell.

In [6]:
MY_SOURCE = 'path/to/your/video.mp4'  # change to your actual image/video path
if MY_SOURCE != 'path/to/your/video.mp4':
    model.predict(source=MY_SOURCE, save=True, conf=0.5, project='runs/detect', name='my_test', exist_ok=True)
    print('Saved predictions under:', Path('runs/detect/my_test').resolve())
else:
    print('Set MY_SOURCE to your actual image/video path to run this optional test.')

Set MY_SOURCE to your actual image/video path to run this optional test.
